# 03 — Patient Profile Parser Agent: Prompt Development

Goal: build a patient profile extraction agent that converts free-text clinical descriptions into validated structured data using llama3 via Ollama.

**Iteration log:**
- Attempt 1 (zero-shot): works but returns prose + JSON + `// comments` = unparseable 30% of the time, latency 24.7s
- Attempt 2 (few-shot + strict JSON-only): clean JSON output, latency ~4s, ICD-10 codes now populated
- Key finding: the model needs format examples more than it needs instructions

**Known limitations after testing:**
- ICD-10 codes approximate, not always exact (AML returned C93.90 instead of C92.00)
- Blood cancer metastatic status unreliable (AML marked metastatic=true — blood cancers don't stage that way)
- FLT3-ITD missing from initial 6-biomarker schema — added in final implementation
- 'some prior treatment' → prior_treatment_lines=1 (reasonable inference but not explicit)

In [ ]:
import requests
import json
import re
import time

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3"

def call_ollama(prompt: str, temperature: float = 0) -> str:
    r = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    })
    return r.json()["response"]

def extract_json(text: str) -> dict | None:
    """Pull the first JSON object from LLM output. Strip JS // comments before parsing."""
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    raw = re.sub(r'//[^\n]*', '', m.group())
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None

print(f"Using model: {MODEL}")

## Attempt 1 — Zero-shot (what failed)

In [ ]:
# This approach fails: model wraps JSON in prose, adds // comments, doesn't know ICD-10
zero_shot_prompt = '''Extract clinical information from this patient description and return ONLY valid JSON.

Patient: 58-year-old female with metastatic triple-negative breast cancer, BRCA2 mutation positive, 2 prior chemotherapy lines, ECOG 1.

Return JSON with these exact keys:
{
  "cancer_type": "...",
  "icd10_code": "...",
  "biomarkers": {"BRCA1": null, "BRCA2": null, "PD_L1": null, "KRAS": null, "HER2": null, "EGFR": null},
  "ecog": null,
  "prior_treatment_lines": null,
  "age": null,
  "metastatic": null
}'''

t0 = time.time()
raw = call_ollama(zero_shot_prompt)
print(f"Latency: {time.time()-t0:.1f}s")
print("Raw output:")
print(raw)
print()
parsed = extract_json(raw)
print(f"Parsed successfully: {parsed is not None}")
if parsed:
    print(json.dumps(parsed, indent=2))

**Why it failed:** Model adds prose before/after JSON, and uses `// comments` inside the object (invalid JSON). ICD-10 code is empty string. Latency is 24s because the model generates a long explanation.

Fix: give it format examples, not just instructions.

## Attempt 2 — Few-shot with ICD-10 examples (what worked)

In [ ]:
FEW_SHOT_PROMPT = '''You are a clinical NLP system. Extract structured patient data from free text.
Return ONLY a JSON object — no explanation, no markdown, no comments.
Use null for any field not mentioned. Use ICD-10 codes for cancer type.

BIOMARKER VALUES must be one of: "positive", "negative", "mutant", "wildtype", or null.

EXAMPLE 1:
Input: 45-year-old male with stage IV non-small cell lung cancer, EGFR exon 19 deletion, never smoked, no prior systemic therapy, ECOG 0.
Output: {{"cancer_type": "Non-small cell lung cancer", "icd10_code": "C34.90", "biomarkers": {{"BRCA1": null, "BRCA2": null, "PD_L1": null, "KRAS": null, "HER2": null, "EGFR": "mutant", "FLT3": null}}, "ecog": 0, "prior_treatment_lines": 0, "age": 45, "metastatic": true}}

EXAMPLE 2:
Input: 62-year-old female with HR-positive HER2-negative metastatic breast cancer, CDK4/6 inhibitor refractory, PIK3CA mutation, ECOG 1, 2 prior lines.
Output: {{"cancer_type": "Breast cancer, HR-positive HER2-negative", "icd10_code": "C50.919", "biomarkers": {{"BRCA1": null, "BRCA2": null, "PD_L1": null, "KRAS": null, "HER2": "negative", "EGFR": null, "FLT3": null}}, "ecog": 1, "prior_treatment_lines": 2, "age": 62, "metastatic": true}}

Now extract from:
Input: {patient_text}
Output:'''

test_inputs = [
    # Simple
    "58-year-old female with metastatic triple-negative breast cancer, BRCA2 mutation positive, 2 prior chemotherapy lines, ECOG 1.",
    # Complex — multiple biomarkers
    "67-year-old male with metastatic NSCLC, KRAS G12C mutation, PD-L1 TPS 60%, HER2 negative, 1 prior platinum-based chemo, ECOG 1, brain metastases.",
    # AML with FLT3
    "71-year-old male with newly diagnosed AML, FLT3-ITD positive, fit for intensive chemo, ECOG 0, no prior treatment.",
    # Sparse — missing most fields
    "Patient with lung cancer, KRAS G12C, some prior treatment.",
    # Ambiguous metastasis
    "52-year-old female with locally advanced cervical cancer, PD-L1 positive, no prior therapy, ECOG 1.",
]

results = []
for text in test_inputs:
    t0 = time.time()
    prompt = FEW_SHOT_PROMPT.format(patient_text=text)
    raw = call_ollama(prompt)
    elapsed = time.time() - t0
    parsed = extract_json(raw)
    results.append((text, parsed, elapsed))
    print(f"Input: {text[:80]}")
    print(f"Latency: {elapsed:.1f}s | Parsed: {parsed is not None}")
    if parsed:
        print(json.dumps(parsed, indent=2))
    else:
        print(f"PARSE FAILED. Raw: {raw[:200]}")
    print()

## Analysis — what worked and what didn't

In [ ]:
print("EXTRACTION ACCURACY SUMMARY")
print("=" * 60)

# Known correct values for manual verification
expected = [
    {"age": 58, "ecog": 1, "prior_treatment_lines": 2, "metastatic": True, "BRCA2": "positive"},
    {"age": 67, "ecog": 1, "prior_treatment_lines": 1, "metastatic": True, "KRAS": "mutant", "PD_L1": "positive", "HER2": "negative"},
    {"age": 71, "ecog": 0, "prior_treatment_lines": 0, "FLT3": "positive"},
    {"age": None, "ecog": None, "KRAS": "mutant"},
    {"age": 52, "ecog": 1, "prior_treatment_lines": 0, "metastatic": False, "PD_L1": "positive"},
]

for i, (text, parsed, elapsed) in enumerate(results):
    print(f"\nCase {i+1}: {text[:60]}...")
    if not parsed:
        print("  FAILED TO PARSE")
        continue
    exp = expected[i]
    hits, misses = [], []
    for field, expected_val in exp.items():
        if field in ["BRCA1","BRCA2","PD_L1","KRAS","HER2","EGFR","FLT3"]:
            actual = parsed.get("biomarkers", {}).get(field)
        else:
            actual = parsed.get(field)
        if actual == expected_val:
            hits.append(field)
        else:
            misses.append(f"{field}: expected={expected_val}, got={actual}")
    print(f"  Correct: {hits}")
    if misses:
        print(f"  Wrong:   {misses}")

## Key findings

**What works reliably:**
- Age, ECOG, prior treatment line counts — extracted correctly on all test cases
- Clearly stated biomarkers (BRCA2 positive, KRAS G12C, HER2 negative) — correct every time
- Metastatic status for solid tumors — correct when explicitly stated
- Latency with few-shot: ~4s vs 25s zero-shot (6x speedup from format guidance)

**Known failure modes:**
- ICD-10 codes approximate — C92.00 (AML) returns as C93.90 occasionally. Use as soft signal only.
- Blood cancer `metastatic` field unreliable — AML/lymphoma don't use solid tumor staging. Downstream agent should treat `metastatic=null` for hematologic malignancies.
- FLT3 missing from schema without explicit addition — added in final implementation.
- 'some prior treatment' → model infers 1 line. Reasonable but should be flagged as inferred.
- 'locally advanced' correctly maps to metastatic=False for cervical cancer.

**Design decision for parser_agent.py:** Run the extraction, validate with Pydantic, and tag any field where the model had to infer rather than directly read from text.

## Pydantic schema validation

In [ ]:
from pydantic import BaseModel, field_validator
from typing import Optional, Literal

BiomarkerValue = Optional[Literal["positive", "negative", "mutant", "wildtype"]]

class BiomarkerStatus(BaseModel):
    BRCA1: BiomarkerValue = None
    BRCA2: BiomarkerValue = None
    PD_L1: BiomarkerValue = None
    KRAS: BiomarkerValue = None
    HER2: BiomarkerValue = None
    EGFR: BiomarkerValue = None
    FLT3: BiomarkerValue = None

class PatientProfile(BaseModel):
    cancer_type: Optional[str] = None
    icd10_code: Optional[str] = None
    biomarkers: BiomarkerStatus = BiomarkerStatus()
    ecog: Optional[int] = None
    prior_treatment_lines: Optional[int] = None
    age: Optional[int] = None
    metastatic: Optional[bool] = None

    @field_validator("ecog")
    @classmethod
    def ecog_range(cls, v):
        if v is not None and not (0 <= v <= 4):
            raise ValueError(f"ECOG must be 0-4, got {v}")
        return v

    @field_validator("prior_treatment_lines")
    @classmethod
    def lines_non_negative(cls, v):
        if v is not None and v < 0:
            raise ValueError(f"prior_treatment_lines cannot be negative, got {v}")
        return v

# Test schema on all parsed results
for text, parsed, _ in results:
    if not parsed:
        print(f"SKIP (parse failed): {text[:50]}")
        continue
    try:
        # Handle biomarkers dict — map PD-L1 key variant if present
        bm_raw = parsed.get("biomarkers", {})
        bm_raw["PD_L1"] = bm_raw.pop("PD-L1", bm_raw.get("PD_L1"))
        profile = PatientProfile(
            cancer_type=parsed.get("cancer_type"),
            icd10_code=parsed.get("icd10_code"),
            biomarkers=BiomarkerStatus(**{k: v for k, v in bm_raw.items() if k in BiomarkerStatus.model_fields}),
            ecog=parsed.get("ecog"),
            prior_treatment_lines=parsed.get("prior_treatment_lines"),
            age=parsed.get("age"),
            metastatic=parsed.get("metastatic"),
        )
        print(f"OK: {text[:60]}")
        print(f"   cancer={profile.cancer_type}, ecog={profile.ecog}, age={profile.age}, meta={profile.metastatic}")
        print(f"   biomarkers={profile.biomarkers.model_dump(exclude_none=True)}")
    except Exception as e:
        print(f"VALIDATION ERROR: {text[:60]}")
        print(f"   {e}")
    print()